# 參考文獻


1.   https://www.toolify.ai/tw/ai-news-tw/llm-%E4%B8%AD%E7%9A%84%E6%BA%AB%E5%BA%A6top-ptop-k-%E6%98%AF%E4%BB%80%E9%BA%BC%E5%BE%9E%E6%A6%82%E5%BF%B5%E5%88%B0%E4%BB%A3%E7%A2%BC-968155
2.   



# Temperature

In [ ]:
import numpy as np

def softmax(logits, temperature=1.0):
    logits = np.array(logits, dtype=float)
    logits = logits / temperature
    exp_logits = np.exp(logits - np.max(logits))  # 防止 overflow
    return exp_logits / np.sum(exp_logits)

def temperature_sampling(logits, words, temperature=1.0, print_words=False):
    logits = np.array(logits, dtype=float)

    # 🔴 特別處理 temperature = 0
    if temperature == 0:
        max_index = np.argmax(logits)
        if print_words:
            for i, word in enumerate(words):
                print(f"{word}: {'1.00' if i == max_index else '0.00'}")
        return words[max_index]

    probabilities = softmax(logits, temperature)

    if print_words:
        for word, probability in zip(words, probabilities):
            print(f"{word}: {probability:.2f}")

    return np.random.choice(words, p=probabilities)

logits = [1, 2, 3, 4]
words = ['A', 'B', 'C', 'D']

temperature_0 = temperature_sampling(logits, words, temperature=0, print_words=True)
temperature_0_9 = temperature_sampling(logits, words, temperature=0.9, print_words=True)

print(f"Temperature 0: {temperature_0}")
print(f"Temperature 0.9: {temperature_0_9}")


# Top-P

In [ ]:
import numpy as np

def softmax(logits, temperature=1.0):
    logits = np.array(logits, dtype=float)
    logits = logits / temperature
    exp_logits = np.exp(logits - np.max(logits))
    return exp_logits / np.sum(exp_logits)

def top_p_sampling(logits, words, top_p=0.9, temperature=1.0, print_words=False):
    logits = np.array(logits, dtype=float)

    probabilities = softmax(logits, temperature)

    sorted_indexes = np.argsort(probabilities)[::-1]
    sorted_probs = probabilities[sorted_indexes]
    cumulative_probs = np.cumsum(sorted_probs)

    # 保留 cumulative prob >= top_p 前的 token（至少一個）
    cutoff = np.searchsorted(cumulative_probs, top_p) + 1
    selected_indexes = sorted_indexes[:cutoff]

    selected_words = [words[i] for i in selected_indexes]
    selected_logits = logits[selected_indexes]

    final_probabilities = softmax(selected_logits, temperature)

    if print_words:
        for word, probability in zip(selected_words, final_probabilities):
            print(f"{word}: {probability:.2f}")

    return np.random.choice(selected_words, p=final_probabilities)


# 🔽 範例
logits = [1, 2, 3, 4]
words = ['A', 'B', 'C', 'D']

result = top_p_sampling(logits, words, top_p=0.9, temperature=1.0, print_words=True)
print(f"Top-P result: {result}")


# Top-K

In [ ]:
import numpy as np

def softmax(logits, temperature=1.0):
    logits = np.array(logits, dtype=float)
    logits = logits / temperature
    exp_logits = np.exp(logits - np.max(logits))
    return exp_logits / np.sum(exp_logits)

def top_k_sampling(logits, words, top_k=3, temperature=1.0, print_words=False):
    logits = np.array(logits, dtype=float)

    probabilities = softmax(logits, temperature)

    sorted_indexes = np.argsort(probabilities)[::-1]
    selected_indexes = sorted_indexes[:top_k]

    selected_words = [words[i] for i in selected_indexes]
    selected_logits = logits[selected_indexes]

    final_probabilities = softmax(selected_logits, temperature)

    if print_words:
        for word, probability in zip(selected_words, final_probabilities):
            print(f"{word}: {probability:.2f}")

    return np.random.choice(selected_words, p=final_probabilities)


# 🔽 範例
logits = [1, 2, 3, 4]
words = ['A', 'B', 'C', 'D']

result = top_k_sampling(logits, words, top_k=3, temperature=1.0, print_words=True)
print(f"Top-K result: {result}")


# Top-P & Top-K

In [ ]:
import numpy as np

def softmax(logits):
    logits = np.array(logits, dtype=float)
    exp_logits = np.exp(logits - np.max(logits))
    return exp_logits / np.sum(exp_logits)

def top_p_sampling(logits, words, top_p, print_words=False):
    probabilities = softmax(logits)

    sorted_indexes = np.argsort(probabilities)[::-1]
    cumulative_probabilities = np.cumsum(probabilities[sorted_indexes])

    selected_indexes = sorted_indexes[cumulative_probabilities <= top_p]

    # 保證至少選到一個
    if len(selected_indexes) == 0:
        selected_indexes = sorted_indexes[:1]

    selected_words = [words[i] for i in selected_indexes]
    softmax_probabilities = softmax(np.array(logits)[selected_indexes])

    if print_words:
        for word, probability in zip(selected_words, softmax_probabilities):
            print(f"{word}: {probability:.2f}")

    return np.random.choice(selected_words, p=softmax_probabilities)

def top_k_sampling(logits, words, top_k, print_words=False):
    probabilities = softmax(logits)

    sorted_indexes = np.argsort(probabilities)[::-1]
    selected_indexes = sorted_indexes[:top_k]

    selected_words = [words[i] for i in selected_indexes]
    softmax_probabilities = softmax(np.array(logits)[selected_indexes])

    if print_words:
        for word, probability in zip(selected_words, softmax_probabilities):
            print(f"{word}: {probability:.2f}")

    return np.random.choice(selected_words, p=softmax_probabilities)


# 🔽 一定要先定義
logits = [1, 2, 3, 4]
words = ['A', 'B', 'C', 'D']

top_p = 0.9
top_k = 3

print("Top-P result:", top_p_sampling(logits, words, top_p, print_words=True))
print("Top-K result:", top_k_sampling(logits, words, top_k, print_words=True))


# Demo Top-K (1~10)

In [ ]:
import numpy as np

def softmax(logits):
    logits = np.array(logits, dtype=float)
    exp_logits = np.exp(logits - np.max(logits))
    return exp_logits / np.sum(exp_logits)

def top_k_sampling(logits, words, top_k, print_words=False):
    probabilities = softmax(logits)

    sorted_indexes = np.argsort(probabilities)[::-1]
    selected_indexes = sorted_indexes[:top_k]

    selected_words = [words[i] for i in selected_indexes]
    selected_logits = np.array(logits)[selected_indexes]
    final_probabilities = softmax(selected_logits)

    if print_words:
        print(f"\nTop-K = {top_k}")
        for word, prob in zip(selected_words, final_probabilities):
            print(f"{word}: {prob:.2f}")

    return np.random.choice(selected_words, p=final_probabilities)


# 🔽 模擬模型的候選 token
words = [
    "好。",
    "適合散步。",
    "有點冷。",
    "很悶。",
    "讓人想睡覺。",
    "適合待在家。",
    "不太穩定。",
    "令人心情愉快。",
    "有點潮濕。",
    "非常舒服。"
]

# 假想 logits（越大代表模型越偏好）
logits = [5.0, 4.5, 3.8, 3.5, 3.2, 3.0, 2.5, 2.3, 2.0, 1.8]

# 🔽 不同 Top-K Demo
for k in [1, 3, 5, 10]:
    result = top_k_sampling(logits, words, top_k=k, print_words=True)
    print(f"👉 回答：今天天氣真{result}")


# Demo Top-K(1~50)

In [ ]:
import numpy as np
import random

# -----------------------------
# Softmax
# 將 logits 轉換成機率分布
# -----------------------------
def softmax(logits):
    # 確保為 float array
    logits = np.array(logits, dtype=float)

    # 減去最大值是常見技巧，避免 exp 爆掉（數值穩定）
    exp_logits = np.exp(logits - np.max(logits))

    # 正規化成機率（總和 = 1）
    return exp_logits / np.sum(exp_logits)


# -----------------------------
# Top-K Sampling
# 只允許機率最高的 K 個 token 參與抽樣
# -----------------------------
def top_k_sampling(logits, words, top_k, print_words=False):
    # 1️⃣ 將 logits 轉成整體機率分布
    probabilities = softmax(logits)

    # 2️⃣ 依機率由大到小排序 token index
    sorted_indexes = np.argsort(probabilities)[::-1]

    # 3️⃣ 只保留前 K 個 token
    selected_indexes = sorted_indexes[:top_k]

    # 4️⃣ 取出 Top-K 對應的文字與 logits
    selected_words = [words[i] for i in selected_indexes]
    selected_logits = np.array(logits)[selected_indexes]

    # 5️⃣ 對 Top-K token 重新做 softmax
    #     （因為其他 token 已被捨棄，機率要重新正規化）
    final_probabilities = softmax(selected_logits)

    # 6️⃣ Demo 用：印出 Top-K token 與其最終機率
    if print_words:
        print(f"\n===== Top-K = {top_k} =====")
        # 只印前 10 個，避免畫面太亂
        for word, prob in zip(selected_words[:10], final_probabilities[:10]):
            print(f"{word}: {prob:.3f}")
        if len(selected_words) > 10:
            print("...")

    # 7️⃣ 根據 Top-K 機率進行隨機抽樣
    return random.choices(
        selected_words,
        weights=final_probabilities,
        k=1
    )[0]


# -----------------------------
# Demo 情境設定
# Prompt:「今天天氣真...」
# -----------------------------

# 模擬模型「可能接續的詞」
# 數量刻意拉到 50 個，用來展示 Top-K 差異
words = [
    "好。", "適合散步。", "有點冷。", "很悶。", "非常舒服。",
    "讓人想睡覺。", "適合待在家。", "令人心情愉快。",
    "有點潮濕。", "不太穩定。",
    "適合出門。", "需要帶傘。", "晴朗。",
    "陰陰的。", "很有秋天的感覺。",
    "適合喝咖啡。", "適合慢跑。", "讓人放鬆。",
    "有點熱。", "溫度剛剛好。",
    "很清爽。", "適合旅行。", "有點悶熱。",
    "很舒適。", "讓人心情平靜。",
    "適合看書。", "適合聽音樂。",
    "空氣不錯。", "有點乾燥。",
    "適合拍照。", "有點涼爽。",
    "讓人精神很好。", "很有夏天的感覺。",
    "讓人想出門走走。",
    "很適合約會。",
    "讓人心情輕鬆。",
    "適合家庭出遊。",
    "有點灰灰的。",
    "很舒服。",
    "適合野餐。",
    "讓人不太想動。",
    "適合工作。",
    "適合放假。",
    "有點多雲。",
    "不會太熱。",
    "不會太冷。",
    "讓人感覺安心。",
    "很宜人。",
    "適合放慢步調。"
]

# 模擬模型內部 logits
# 前面分數高、後面低，符合真實語言模型的分布特性
logits = np.linspace(8.0, 1.0, len(words))


# -----------------------------
# 不同 Top-K 的 Demo
# -----------------------------
for k in [1, 10, 25, 50]:
    result = top_k_sampling(logits, words, top_k=k, print_words=True)
    print(f"👉 回答：今天天氣真__{result}")



===== Top-K = 1 =====
好。: 1.000
👉 回答：今天天氣真__好。

===== Top-K = 10 =====
好。: 0.177
適合散步。: 0.153
有點冷。: 0.132
很悶。: 0.114
非常舒服。: 0.099
讓人想睡覺。: 0.085
適合待在家。: 0.074
令人心情愉快。: 0.064
有點潮濕。: 0.055
不太穩定。: 0.048
👉 回答：今天天氣真__好。

===== Top-K = 25 =====
好。: 0.139
適合散步。: 0.120
有點冷。: 0.104
很悶。: 0.090
非常舒服。: 0.078
讓人想睡覺。: 0.067
適合待在家。: 0.058
令人心情愉快。: 0.050
有點潮濕。: 0.043
不太穩定。: 0.038
...
👉 回答：今天天氣真__不太穩定。

===== Top-K = 50 =====
好。: 0.136
適合散步。: 0.117
有點冷。: 0.101
很悶。: 0.088
非常舒服。: 0.076
讓人想睡覺。: 0.066
適合待在家。: 0.057
令人心情愉快。: 0.049
有點潮濕。: 0.042
不太穩定。: 0.037
...
👉 回答：今天天氣真__晴朗。


# Demo Top-K(1~50) Run 3

In [ ]:
import numpy as np
import random

from datetime import datetime
import time

# -----------------------------
# Softmax
# 將 logits 轉換成機率分布
# -----------------------------
def softmax(logits):
    # 確保為 float array
    logits = np.array(logits, dtype=float)

    # 減去最大值是常見技巧，避免 exp 爆掉（數值穩定）
    exp_logits = np.exp(logits - np.max(logits))

    # 正規化成機率（總和 = 1）
    return exp_logits / np.sum(exp_logits)


# -----------------------------
# Top-K Sampling
# 只允許機率最高的 K 個 token 參與抽樣
# -----------------------------
def top_k_sampling(logits, words, top_k, print_words=False):
    # 1️⃣ 將 logits 轉成整體機率分布
    probabilities = softmax(logits)

    # 2️⃣ 依機率由大到小排序 token index
    sorted_indexes = np.argsort(probabilities)[::-1]

    # 3️⃣ 只保留前 K 個 token
    selected_indexes = sorted_indexes[:top_k]

    # 4️⃣ 取出 Top-K 對應的文字與 logits
    selected_words = [words[i] for i in selected_indexes]
    selected_logits = np.array(logits)[selected_indexes]

    # 5️⃣ 對 Top-K token 重新做 softmax
    #     （因為其他 token 已被捨棄，機率要重新正規化）
    final_probabilities = softmax(selected_logits)

    # 6️⃣ Demo 用：印出 Top-K token 與其最終機率
    if print_words:
        print(f"\n===== Top-K = {top_k} =====")
        # 只印前 10 個，避免畫面太亂
        for word, prob in zip(selected_words[:10], final_probabilities[:10]):
            print(f"{word}: {prob:.3f}")
        if len(selected_words) > 10:
            print("...")

    # 7️⃣ 根據 Top-K 機率進行隨機抽樣
    return random.choices(
        selected_words,
        weights=final_probabilities,
        k=1
    )[0]


# -----------------------------
# Demo 情境設定
# Prompt:「今天天氣真...」
# -----------------------------

# 模擬模型「可能接續的詞」
# 數量刻意拉到 50 個，用來展示 Top-K 差異
words = [
    # -----------------
    # 高合理性（非常正常）
    # -----------------
    "好。", "非常舒服。", "很宜人。", "溫度剛剛好。",
    "適合散步。", "適合出門。", "很清爽。",
    "讓人心情愉快。", "空氣不錯。", "很舒適。",

    # -----------------
    # 中等合理性（稍微怪）
    # -----------------
    "讓人有點想睡覺。", "適合一直喝咖啡。",
    "陰陰的但又有點熱。", "有點悶又有點冷。",
    "讓人不知道該穿什麼。", "好像要下雨但又沒下。",
    "適合出門但又懶得出門。",
    "讓人想待在家又怕無聊。",
    "天氣有點在猶豫。", "不太穩定。",

    # -----------------
    # 低合理性（明顯不自然）
    # -----------------
    "讓人突然想辭職。", "像冷氣壞掉一樣。",
    "感覺天空在發呆。", "今天的天氣沒有意義。",
    "讓人對人生產生疑問。",
    "很像星期一的感覺。",
    "適合對著天空發呆。",
    "空氣彷彿在偷懶。",
    "天氣好像不想工作。",
    "讓人什麼都不想做。",

    # -----------------
    # 極低合理性（Demo 用異常）
    # -----------------
    "今天天氣真 404 Not Found。",
    "天氣進入待機模式。",
    "雲好像當機了。",
    "大氣層需要重新開機。",
    "今天天氣真 undefined。",
    "系統回應異常。",
    "請稍後再試。",
    "天氣正在更新中。",
    "發生未知錯誤。",
    "今天天氣真 ???"
]

# 模擬模型內部 logits
# 前面分數高、後面低，符合真實語言模型的分布特性
logits = np.linspace(5.0, 2.0, len(words))


# -----------------------------
# Top-K 多次抽樣 Demo
# -----------------------------
# 每個 Top-K 抽樣次數

num_runs = 3

# -----------------------------
# 不同 Top-K 的 Demo（每個 K 跑 3 次）
# -----------------------------
num_runs = 3

for k in [1, 10, 25, 50]:
    print(f"\n==============================")
    print(f" Top-K = {k}")
    print(f"==============================")

    for run in range(1, num_runs + 1):
        timestamp = datetime.now().strftime("%H:%M:%S")

        # 👉 只有第一次 run 才印出 Top-10 機率分布（解釋用）
        show_distribution = (run == 1)

        result = top_k_sampling(
            logits=logits,
            words=words,
            top_k=k,
            print_words=show_distribution
        )

        print(f"[{timestamp}] Run {run}: 今天天氣真__{result}")

        # 讓時間錯開一點，比較好對照
        time.sleep(2)


 Top-K = 1

===== Top-K = 1 =====
好。: 1.000
[03:01:16] Run 1: 今天天氣真__好。
[03:01:18] Run 2: 今天天氣真__好。
[03:01:20] Run 3: 今天天氣真__好。

 Top-K = 10

===== Top-K = 10 =====
好。: 0.138
非常舒服。: 0.128
很宜人。: 0.118
溫度剛剛好。: 0.110
適合散步。: 0.101
適合出門。: 0.094
很清爽。: 0.087
讓人心情愉快。: 0.081
空氣不錯。: 0.075
很舒適。: 0.069
[03:01:22] Run 1: 今天天氣真__適合散步。
[03:01:24] Run 2: 今天天氣真__非常舒服。
[03:01:26] Run 3: 今天天氣真__溫度剛剛好。

 Top-K = 25

===== Top-K = 25 =====
好。: 0.087
非常舒服。: 0.080
很宜人。: 0.074
溫度剛剛好。: 0.069
適合散步。: 0.064
適合出門。: 0.059
很清爽。: 0.055
讓人心情愉快。: 0.051
空氣不錯。: 0.047
很舒適。: 0.043
...
[03:01:28] Run 1: 今天天氣真__空氣不錯。
[03:01:30] Run 2: 今天天氣真__讓人對人生產生疑問。
[03:01:32] Run 3: 今天天氣真__很舒適。

 Top-K = 50

===== Top-K = 50 =====
好。: 0.078
非常舒服。: 0.072
很宜人。: 0.067
溫度剛剛好。: 0.062
適合散步。: 0.057
適合出門。: 0.053
很清爽。: 0.049
讓人心情愉快。: 0.045
空氣不錯。: 0.042
很舒適。: 0.039
...
[03:01:34] Run 1: 今天天氣真__適合一直喝咖啡。
[03:01:36] Run 2: 今天天氣真__不太穩定。
[03:01:38] Run 3: 今天天氣真__適合一直喝咖啡。
